In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import torch
from torch import nn

from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms

import matplotlib.pyplot as plt

In [ ]:
os.listdir(os.path.join(path,"PlantVillage"))

In [ ]:
# Write your code here
train_path = os.path.join(path,"PlantVillage","train")
test_path = os.path.join(path,"PlantVillage","test")
train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.Resize((32,32)),
    transforms.ToTensor()
])
test_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor()
])

In [ ]:
train_dataset = ImageFolder(train_path,transform=train_transform)
test_dataset =  ImageFolder(train_path,transform=test_transform)

In [ ]:
train_dataset.classes

In [ ]:
train_loader = DataLoader(train_dataset,16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset,16, shuffle=True, num_workers=2)

In [ ]:
classes= train_dataset.classes
images, lable = next(iter(train_loader))
for i in range(6):
  plt.subplot(2,3,i+1)
  plt.imshow(images[i].permute(1,2,0))
  plt.title(classes[lable[i]],size=8)
  plt.axis("off")
  print(images[i].shape)


In [ ]:
# Write your code here
model = nn.Sequential(
    nn.Conv2d(3,16,3),
    nn.BatchNorm2d(16),
    nn.ReLU(),
    nn.Conv2d(16,32,3),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.Conv2d(32,64,3),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.Conv2d(64,128,3),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.Conv2d(128,128,3),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(22*22*128,3)
)

model

In [ ]:
# Write your code here
# Write your code here
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    t = 1
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)  # The model outputs in shape [batch_size,1]. We convert them to [batch_size,] so the loss accepts them.
        loss = criterion(outputs, labels)


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Track accuracy
        l= torch.softmax(outputs,dim=1)
        predictions = torch.argmax(l, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # The model outputs in shape [batch_size,1]. We convert them to [batch_size,] so the loss accepts them.
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Compute accuracy
            l= torch.softmax(outputs,dim=1)
            predictions = torch.argmax(l, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


In [ ]:
def train(model, train_loader,valid_loader,criterion,optimizer,num_epochs ,device):
# Lists to store metrics
  train_losses = []
  val_losses = []
  train_accuracies = []
  val_accuracies = []

  # Training process
  for epoch in tqdm(range(num_epochs)):
      train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
      val_loss, val_accuracy = validate(model, valid_loader, criterion, device)

      # Store metrics
      train_losses.append(train_loss)
      val_losses.append(val_loss)
      train_accuracies.append(train_accuracy)
      val_accuracies.append(val_accuracy)

      if epoch % 2:
        print(f"Epoch {epoch+1}/{num_epochs}: "
              f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
              f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")
  return train_losses,val_losses,train_accuracies,val_accuracies

In [ ]:
# Write your code here
device= "cuda" if torch.cuda.is_available() else "cpu"
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs
model.to(device)

device

In [ ]:
train_losses,val_losses,train_accuracies,val_accuracies = train(model,train_loader, test_loader, criterion ,optimizer ,num_epochs , device)

In [ ]:
plt.plot(train_losses, c="r",label="train")
plt.plot(val_losses ,c="b",label= "test")
plt.title("train loss vs test loss")
plt.legend()

In [ ]:
plt.plot(train_accuracies, c= "r", label="train")
plt.plot(val_accuracies, c="b", label= "test")
plt.title("train accuracy vs test accuracy")
plt.legend()

In [ ]:
# Write your code here
